In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
# Create a lookup table:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [8]:
!pipenv install toyaikit

Loading .env environment variables...
Courtesy Notice:
Pipenv found itself running within a virtual environment,  so it will 
automatically use that environment, instead of  creating its own for any 
project. You can set
PIPENV_IGNORE_VIRTUALENVS=1 to force pipenv to ignore that environment and 
create  its own instead.
You can set PIPENV_VERBOSITY=-1 to suppress this warning.
To activate this project's virtualenv, run pipenv shell.
Alternatively, run a command inside the virtualenv with pipenv run.
Installing toyaikit...
✔ Installation Succeeded
To activate this project's virtualenv, run pipenv shell.
Alternatively, run a command inside the virtualenv with pipenv run.
Installing dependencies from Pipfile.lock (f18ca5)...
All dependencies are now up-to-date!
Upgrading toyaikit in  dependencies.
Building requirements...
Resolving dependencies....
✔ Success! Locking packages...
⠇ Locking packages...Warning: WARNING:pipenv.patched.pip._vendor.urllib3.connectionpool:Connection pool is full

In [4]:
# Running the agent

from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient


import os 

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
llm_model = "openai/gpt-oss-20b"

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [13]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

# runner = OpenAIResponsesRunner(
#     tools=agent_tools,
#     developer_prompt=instructions,
#     llm_client=OpenAIClient(model=llm_model)
# )

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt = instructions,
    llm_client=OpenAIClient(
        model=llm_model,
        client=OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY")
        )))




In [14]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])


/home/hsu/.local/share/virtualenvs/module_4-8aqFlWUe/lib/python3.10/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-20b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


The result contains:

last_message: the final response
all_messages: the full message history
cost: the cost of all LLM calls in this run

In [15]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it okay to join the course late if I just found it now?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01ky761bs4frs94dx100r3xhwd', summary=[], type='reasoning', content=[Content(text='We need to answer based on FAQ search results. We need to search for relevant FAQ entry.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join the course late if I just found it now?"}', call_id='fc_7ed95c45-99cd-4096-a61c-dd3a3886ff51', name='search', type='function_call', id='fc_7ed95c45-99cd-4096-a61c-dd3a3886ff51', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_7ed95c45-99cd-4096-a61c-dd3a3886ff51',
  'output': '[\n  {\n    "

In [16]:
# Extract the function name and arguments:

def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [17]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join the course late if I just found it now?"}'}]

In [18]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [19]:
# Save the A->Q->A' record and the trajectory:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
   # "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_agent': 'Yes! You can jump in at any time.  \nIf you want to earn a certificate, make sure to submit your Capstone project while the submission window is still open. Otherwise you’re free to start learning and working through the material right away—no registration or prior enrollment is required.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join the course late if I just found it now?"}'}],
 'document': '74eb249bbf'}

In [20]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

In [ ]:
df_agent = pd.DataFrame(agent_answers)

In [ ]:
# Calculate the total cost:
df_agent["cost"].sum()

In [ ]:
df_agent.to_csv("data/agent-answers.csv", index=False)


In [1]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget -O data/agent-answers.csv ${PREFIX}/04-evaluation/data/agent-answers.csv
!cp /home/hsu/Documents/Learning/llm-zoomcamp/04-evaluation/data/agent-answers.csv ./data/agent-answers.csv 

In [21]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Our judge can look at both:

- whether answer_agent matches answer_orig
- whether the tool calls look reasonable for the question

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [23]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [24]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [25]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0],llm_model)

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer correctly states that a self‑paced enrollment does not receive a certificate and that a certificate is only awarded for a live cohort, noting the need for peer review during the course. This matches the key information in the original answer.', answer_score='good', trajectory_reasoning='The single search query included the essential keywords from the question (self‑paced, certificate, course) and was aimed at confirming the policy. No duplicate or unnecessary calls were made, and the number of calls (one) is reasonable. The retrieved information supports the final answer.', trajectory_score='good')

In [26]:
# Run the judge for all agent answers:

def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage


In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [ ]:
# Calculate the judge cost from the token usage:

calc_total_price(usages)

In [ ]:
# Check the answer scores:

df_agent_eval["answer_score"].value_counts()

In [ ]:
# Check the trajectory scores:

df_agent_eval["trajectory_score"].value_counts()

In [ ]:
# Save the judge results:

df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

In [27]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/agent-evaluations.csv ${PREFIX}/04-evaluation/data/agent-evaluations.csv
!cp /home/hsu/Documents/Learning/llm-zoomcamp/04-evaluation/data/agent-evaluations.csv ./data/agent-evaluations.csv